In [ ]:
!pip install -q google-genai ipywidgets


# NutriPrompt – Gemini Prototype

## Objetivo
Generar planes semanales personalizados bajos en FODMAPs usando IA.

## Qué estoy trabajando
- Prompt Engineering
- Integración con Gemini API
- Generación estructurada (JSON / Markdown)
- Base para integración en Django

## Nota importante
Los planes generados son orientativos y no sustituyen el asesoramiento de un profesional sanitario.

In [1]:
from google.colab import userdata
from google import genai

# Lee tu API key guardada en Colab Secrets
API_KEY = userdata.get("GOOGLE_API_KEY")

# Crea el cliente
client = genai.Client(api_key=API_KEY)

print("Conexión con Gemini preparada correctamente.")

ModuleNotFoundError: No module named 'google.colab'

## 4. System Prompt

In [ ]:
SYSTEM_PROMPT = """
Actúa como un asistente de generación de planes alimenticios inspirado en buenas prácticas nutricionales, con especial atención a la dieta baja en FODMAPs.

Tu objetivo es generar un plan semanal estructurado y orientativo adaptado a un cliente.

IMPORTANTE:
- Este plan es orientativo y no sustituye el asesoramiento de un profesional sanitario.
- No realices afirmaciones médicas ni diagnósticos.

REQUISITOS:
- Genera un plan de 7 días
- Incluye desayuno, comida y cena
- Evita alimentos altos en FODMAPs
- Evita ajo y cebolla
- Usa comidas sencillas
- Mantén variedad

FORMATO DE SALIDA:
Devuelve SOLO un JSON válido con esta estructura:

[
  {
    "day": "Lunes",
    "breakfast": "...",
    "lunch": "...",
    "dinner": "..."
  }
]

No añadas texto fuera del JSON.
"""

## 5. Generar Plan

In [ ]:
user_input = """
Perfil del cliente:
- Mujer de 35 años
- Dieta baja en FODMAPs
- Sin lactosa
- Evitar ajo y cebolla
- Poco tiempo para cocinar
- Presupuesto medio
- Prefiere pollo y pescado
"""

full_prompt = f"{SYSTEM_PROMPT}\n\n{user_input}"

try:
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=full_prompt
    )

    raw_output = response.text

    print("Respuesta de la IA:\n")
    print(raw_output)

except Exception as e:
    print("Error al generar contenido ❌")
    print(type(e).__name__)
    print(e)

Error al generar contenido ❌
ClientError
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\nPlease retry in 49.821704982s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-li

## 6. Validar JSON

In [ ]:
import json

# 🔥 limpiar Markdown
clean_output = raw_output.replace("```json", "").replace("```", "").strip()

print("OUTPUT LIMPIO 👇")
print(clean_output[:200])  # para comprobar

try:
    data = json.loads(clean_output)
    print("JSON válido ✅")
    display(data)
except Exception as e:
    print("Error al parsear JSON ❌")
    print(e)

OUTPUT LIMPIO 👇
[
  {
    "day": "Lunes",
    "breakfast": "Avena (certificada sin gluten) preparada con leche de almendras sin azúcar, rodajas de plátano verde (firme) y un puñado de arándanos frescos.",
    "lunch"
JSON válido ✅


[{'day': 'Lunes',
  'breakfast': 'Avena (certificada sin gluten) preparada con leche de almendras sin azúcar, rodajas de plátano verde (firme) y un puñado de arándanos frescos.',
  'lunch': 'Ensalada grande de espinacas baby con pechuga de pollo a la plancha troceada, pepino en rodajas, zanahoria rallada y un aderezo sencillo de aceite de oliva virgen extra y vinagre de vino tinto.',
  'dinner': 'Salmón al horno con hierbas frescas (como eneldo o perejil), acompañado de patatas asadas en gajos y judías verdes al vapor.'},
 {'day': 'Martes',
  'breakfast': 'Huevos revueltos con espinacas salteadas y una tostada de pan sin gluten.',
  'lunch': 'Restos de salmón y patatas asadas de la cena del lunes.',
  'dinner': 'Pechuga de pollo a la plancha o a la parrilla, servida con arroz blanco cocido y calabacín a la parrilla o salteado.'},
 {'day': 'Miércoles',
  'breakfast': 'Batido de leche de almendras sin azúcar, un puñado de espinacas, fresas y una cucharada de proteína en polvo baja en FOD

## 7. Mostrar en HTML

In [ ]:
from IPython.display import HTML, display
import pandas as pd

df = pd.DataFrame(data)

html_table = df.to_html(index=False)

display(HTML(html_table))

day,breakfast,lunch,dinner
Lunes,"Avena (certificada sin gluten) preparada con leche de almendras sin azúcar, rodajas de plátano verde (firme) y un puñado de arándanos frescos.","Ensalada grande de espinacas baby con pechuga de pollo a la plancha troceada, pepino en rodajas, zanahoria rallada y un aderezo sencillo de aceite de oliva virgen extra y vinagre de vino tinto.","Salmón al horno con hierbas frescas (como eneldo o perejil), acompañado de patatas asadas en gajos y judías verdes al vapor."
Martes,Huevos revueltos con espinacas salteadas y una tostada de pan sin gluten.,Restos de salmón y patatas asadas de la cena del lunes.,"Pechuga de pollo a la plancha o a la parrilla, servida con arroz blanco cocido y calabacín a la parrilla o salteado."
Miércoles,"Batido de leche de almendras sin azúcar, un puñado de espinacas, fresas y una cucharada de proteína en polvo baja en FODMAPs (por ejemplo, proteína de guisante sin aditivos).",Arroz blanco con pollo desmenuzado (puede ser de un pollo asado sin condimentos FODMAPs o cocido en casa) y zanahorias baby cocidas al vapor.,"Tilapia al horno con boniato asado (en porción controlada, aproximadamente 1/2 taza) y pimientos rojos asados."
Jueves,"Tostadas de pan sin gluten con aguacate (en porción controlada, aproximadamente 1/8 de aguacate pequeño) y rodajas de tomate.","Restos de tilapia, boniato y pimientos rojos asados de la cena del miércoles.","Revuelto de huevo con pimientos rojos picados y espinacas frescas, acompañado de una porción de arroz integral (en porción controlada, aproximadamente 1 taza cocida)."
Viernes,Yogur natural sin lactosa con frambuesas frescas y una cucharada de semillas de chía.,"Ensalada de atún (en agua o aceite, bien escurrido) con lechuga, pepino y zanahoria rallada, aderezado con aceite de oliva y zumo de limón.","Pollo salteado con brócoli (solo floretes, en porción controlada, aproximadamente 3/4 de taza) y zanahorias en una salsa ligera de tamari (salsa de soja sin gluten) y jengibre fresco rallado."
Sábado,"Panqueques caseros de plátano (hechos con huevo, harina de avena sin gluten y un plátano maduro), acompañados de un chorrito de sirope de arce puro.","Sopa casera de calabacín y zanahoria (preparada con caldo de pollo bajo en FODMAPs y sin ajo ni cebolla), con un poco de arroz blanco añadido.","Merluza al vapor o a la plancha, servida con quinoa cocida y una ensalada verde sencilla (lechuga, tomate y pepino) con aderezo de aceite y limón."
Domingo,"Revuelto de huevo con espinacas y pimiento rojo, acompañado de una porción de melón cantalupo.",Restos de la sopa casera de calabacín y zanahoria.,"Pollo asado entero o en piezas (muslos, contramuslos) condimentado con romero, tomillo, sal y pimienta, acompañado de patatas baby asadas y calabacín a la parrilla."
